# Woodpile beam-spreading experiment (focused Gaussian beam, exit-face field)

Replicates the LSU "beam spreading vs frequency" numerical experiment on the woodpile photonic crystals in
`../Structures`. A **focused Gaussian beam** is launched onto the entrance face (`z = -t_slab_z/2`) and the
**complex E-field on the exit face** (`z = +t_slab_z/2`) is recorded with a frequency-domain `FieldMonitor`, so that
the transverse intensity `I(x, y; ν)` and its width `σ²(ν)`, `d(ν)` can be analysed afterwards. An identical
entrance-face monitor records the incident spot in the same run, and an **empty-box reference** (`sim0`, same grid,
no structures) is submitted once per structure to normalise the exit field and measure the free-space spot.



 The transverse simulation size is exactly `params["box_size"]`, with the absorbing layers added outside by tidy3d, so any lateral
spreading beyond `±6 µm` is **clipped by the absorbers**. Nothing structure-specific is typed by hand: the loop reads
box size, rods, defects and permittivity from each `*_tables.h5`, so larger structures generated later run unchanged.

**Nothing is submitted unless `run = True`** — the default `run = False` branch only uploads a throw-away copy to get
`web.estimate_cost`, deletes it, and plots the layout.


In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
from tidy3d import web

# AutomationModule lives in the root of the tidy3d project (only read_hdf5_as_dict is used here)
sys.path.append(os.path.abspath(r'H:\codes\tidy3d'))
import AutomationModule as AM

tidy3dAPI = os.environ["API_TIDY3D_KEY"]
web.configure(tidy3dAPI)          # explicit: this notebook does not go through AM.loadAndRunStructure
print("tidy3d", td.__version__)


## Parameters

The only hand-set physics parameters are `lambdas`, `w0`, `runtime_ps`, `min_steps_per_lambda` and `absorbers`.
Everything geometric (`box_size`, `d`, `permittivity`, radii, rod/defect end-points) comes from `params`, `rods` and
`defects` in each structure file.


### Beam waist

`w0 ` (≈ λ at the gap centre, λ ≈ 2.9 µm). The next cell prints the free-space NA ≈ λ/(π w0) and Rayleigh
range z_R = π w0²/λ at the band edges and the gap centre. The waist sits **on the entrance face**: the source plane is
1 µm below it and, in tidy3d 2.9.x, a *positive* `waist_distance` puts the waist **behind** the source, so
`waist_distance = -(z_face - z_src) = -1 µm` (verified numerically at the end of the notebook).

### Grid

`td.GridSpec.auto(min_steps_per_wvl=18, wavelength=lambdas[0], dl_min=dl, max_scale=1.2)` with
`dl = (λ_min/18)/n_rod`, exactly as in the reference. Note that `wavelength=lambdas[0]` is the **longest** wavelength,
so the auto mesher targets 18 steps per λ_max; the resulting step inside the rods (printed below) therefore resolves
λ_min/n_rod with fewer steps (≈10) and the rod diameter with ≈7 cells. This mirrors the reference/transmission settings;
tighten `min_steps_per_lambda` if the analysis turns out to be resolution-sensitive.


In [ ]:
# ---------------- hand-set parameters ----------------
lambdas = 1.2/np.array([0.28,0.38])      # [lambda_max, lambda_min] in um (same ordering as the reference)
w0 = 3                            # Gaussian-beam waist radius (um), waist on the entrance face
runtime_ps = 35e-12                 # run time in seconds (shutoff=1e-20 will end the run earlier if the field decays)
min_steps_per_lambda = 15
absorbers = 110                     # same absorber count as the woodpile transmission run
source_kind = "planewave_patch"            # "gaussian" (default) | "planewave_patch" (reference-like 5x5 um patch)
patch_size = 4.0                    # only for source_kind == "planewave_patch"
src_offset = 1.0                    # source plane sits this far below the entrance face (um)
band_width = 0.01                   # a/lambda band width for the spectral_sampling report
run = True                         # <<< leave False; the user flips it to submit. False = estimate cost + plots only.
ref = False

project_name = "20261007_patch_Beam_Spreading_Experiment_woodpiles"
folder_path = Path("../Structures/Large_Structures/Lx_132.00_Ly_132.00_Lz_15.27")
# folder_path = Path("../Structures/Large_Structures/Lx_120.00_Ly_120.00_Lz_20.36")

f_min, f_max = td.C_0 / lambdas[0], td.C_0 / lambdas[1]
f0 = 0.5 * (f_min + f_max)

# ---------------- beam numbers (free space) ----------------
print(f"\nBand: lambda = [{lambdas[1]}, {lambdas[0]}] um,  f = [{f_min/1e12:.2f}, {f_max/1e12:.2f}] THz")
for lam in (lambdas[1], 2.9, lambdas[0]):
    print(f"  lambda = {lam:.2f} um: NA ~ lambda/(pi w0) = {lam/(np.pi*w0):.3f}, "
          f"z_R = pi w0^2/lambda = {np.pi*w0**2/lam:.2f} um, "
          f"far-field half-angle = {np.degrees(lam/(np.pi*w0)):.1f} deg")


## Helpers

* `create_cylinder_from_ends(top, bottom, radius)` — same name/signature as the reference. Woodpile rods and defect
  segments are axis-aligned, so it returns `td.Cylinder(center=midpoint, radius, length=|p2-p1|, axis=0|1|2)` directly
  (no `td.Transformed`, no rotation matrix). It raises if the end-points differ in more than one coordinate.
* `woodpile_geometry_from_tables(rods, defects, params)` — one cylinder per rod and one per defect segment, using the
  end-points **exactly as stored**. Positive κ (thicker segments) simply overlap the thinner parent rod: a
  `GeometryGroup` is a union, so no rod splitting is needed. Negative κ (thinner/missing segments) would need the parent
  rod split into pieces (`woodpile_helpers._rod_pieces`) — refused with a clear error.
* `spectral_sampling(T_ps, band_width, a, lambdas)` — the reference helper: number of monitor frequencies needed to
  IFFT the field to an unaliased time window ≥ `T_ps` (`dν ≤ 1/T`).


In [ ]:
from spreading_helpers import *

## Simulation builder

`build_simulation(file)` reads one `*_tables.h5` and returns the structure simulation `sim`, the empty-box reference
`sim0` (identical grid, sources, monitors and `run_time`; only `structures=[]`) and a small `info` dict. Layout:

* **Size.** `(t_slab_x, t_slab_y, t_slab_z) = params["box_size"]`; `Lz = t_slab_z + 2·padding` with `padding = λ_max`
  of air on each side; transverse size = `t_slab_x, t_slab_y` (absorber layers are added outside by tidy3d).
* **Boundaries.** `td.Absorber(num_layers=absorbers)` on all six sides — the beam must leave the box laterally.
* **Source.** `td.GaussianBeam` at `z_src = -t_slab_z/2 - 1`, `direction='+'`, `pol_angle=0` (E along x),
  `waist_radius=w0`, `waist_distance=-(z_face - z_src)` (waist on the entrance face), patch `S = min(6 w0, t_slab_x)`.
  `GaussianPulse(freq0=f0, fwidth=0.5·(f_max−f_min), offset=10)`.
* **Monitors.** `"monitorField_exit"` at `z = +t_slab_z/2` and `"monitorField_entry"` at `z = −t_slab_z/2`, both
  `size=(t_slab_x, t_slab_y, 0)`, `Ex, Ey, Ez`, `freqs=monitor_freqs`, `interval_space=(2, 2, 1)`.
* **Grid / run.** `GridSpec.auto(min_steps_per_wvl, wavelength=λ_max, dl_min=(λ_min/steps)/n_rod, max_scale=1.2)`,
  `subpixel=True`, `shutoff=1e-20`, `normalize_index=None`, background `td.Medium(permittivity=1)`.
* **Frequencies.** `nfreqs = spectral_sampling(runtime_ps·1e12, band_width, a=d, lambdas)`,
  `monitor_freqs = linspace(f_min, f_max, nfreqs)`.

**Polarisation note.** `pol_angle=0` puts E along x. Rod `orientation` 0 = rod along x, 1 = rod along y. The cell prints
the orientation of the top (max-z, exit) and bottom (entrance) layers, i.e. whether E is parallel or perpendicular to
the last/first rods (for the current files: bottom layer along x → E ∥ first rods; top layer along y → E ⊥ last rods).


In [ ]:
def make_source(kind, f0, fwidth, z_src, waist_distance, S, w0):
    st = td.GaussianPulse(freq0=f0, fwidth=fwidth, offset=10)
    if kind == "gaussian":
        return td.GaussianBeam(source_time=st, size=(S, S, 0), center=(0, 0, z_src), direction='+',
                               pol_angle=0, waist_radius=w0, waist_distance=waist_distance, name="gaussian_beam")
    elif kind == "planewave_patch":
        return td.PlaneWave(source_time=st, size=(patch_size, patch_size, 0), center=(0, 0, z_src),
                            direction='+', pol_angle=0, name="planewave")
    raise ValueError(f"unknown source_kind {kind!r}")


def build_simulation(file, verbose=True):
    data = AM.read_hdf5_as_dict(str(file))
    params, rods, defects = data["params"], data["rods"], data["defects"]

    t_slab_x, t_slab_y, t_slab_z = [float(v) for v in params["box_size"]]
    n_rod = float(np.sqrt(params["permittivity"]))
    d = float(params["d"])

    structure, n_cyl, n_rods, n_def = woodpile_geometry_from_tables(rods, defects, params)

    # ---- frequencies
    nfreqs = spectral_sampling(T_ps=runtime_ps * 1e12, band_width=band_width, a=d, lambdas=lambdas)
    monitor_freqs = np.linspace(f_min, f_max, nfreqs)

    # ---- geometry of the box
    padding = float(lambdas[0])                    # >= lambda_max of air on each side
    Lz = t_slab_z + 2 * padding
    z_face_in, z_face_out = -t_slab_z / 2, +t_slab_z / 2
    z_src = z_face_in - src_offset
    waist_distance = -(z_face_in - z_src)          # negative: waist in FRONT of the source plane (tidy3d 2.9.x)
    S = float(min(6 * w0, t_slab_x))

    source = make_source(source_kind, f0, 0.5 * (f_max - f_min), z_src, waist_distance, S, w0)

    monitorField_exit = td.FieldMonitor(center=[0, 0, z_face_out], size=[t_slab_x, t_slab_y, 0],
                                        fields=["Ex", "Ey", "Ez"], freqs=monitor_freqs,
                                        interval_space=(2, 2, 1), name="monitorField_exit")
    monitorField_entry = monitorField_exit.updated_copy(center=[0, 0, z_face_in], name="monitorField_entry")

    A = td.Absorber(num_layers=absorbers)
    boundaries = td.BoundarySpec(x=td.Boundary(plus=A, minus=A),
                                 y=td.Boundary(plus=A, minus=A),
                                 z=td.Boundary(plus=A, minus=A))

    dl = (lambdas[1] / min_steps_per_lambda) / n_rod    # smallest wavelength inside the rods
    sim = td.Simulation(
        center=(0, 0, 0),
        size=(t_slab_x, t_slab_y, Lz),
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=min_steps_per_lambda, wavelength=lambdas[0],
                                   dl_min=dl, max_scale=1.2),
        sources=[source],
        monitors=[monitorField_exit],
        run_time=runtime_ps,
        shutoff=1e-20,
        boundary_spec=boundaries,
        normalize_index=None,
        structures=[structure],
        subpixel=True,
        medium=td.Medium(permittivity=1),
    )
    # empty-box reference on the SAME mesh
    sim0 = sim.copy(update={"structures": [], "grid_spec": td.GridSpec.from_grid(sim.grid)})

    # ---- report
    z_layers = np.atleast_1d(rods["z"]); orient = np.atleast_1d(rods["orientation"])
    top_or = int(orient[np.argmax(z_layers)]); bot_or = int(orient[np.argmin(z_layers)])
    axname = {0: "x", 1: "y"}
    dz_in_rods = np.diff(sim.grid.boundaries.z)
    zc = sim.grid.centers.z
    dz_slab = dz_in_rods[(zc > z_face_in) & (zc < z_face_out)]
    info = dict(params=params, rods=rods, defects=defects, n_cyl=n_cyl, n_rods=n_rods, n_def=n_def, nfreqs=nfreqs,
                monitor_freqs=monitor_freqs, d=d, n_rod=n_rod, box=(t_slab_x, t_slab_y, t_slab_z), Lz=Lz,
                z_src=z_src, waist_distance=waist_distance, S=S, dl=dl, top_orientation=top_or, bottom_orientation=bot_or)
    if verbose:
        dnu = monitor_freqs[1] - monitor_freqs[0]
        print(f"  box = {t_slab_x} x {t_slab_y} x {t_slab_z:.4f} um, Lz = {Lz:.3f} um, n_rod = {n_rod}, d = {d} um")
        print(f"  source '{source_kind}' at z_src = {z_src:.3f} um, waist_distance = {waist_distance:+.2f} um "
              f"(waist at z = {z_src - waist_distance:.3f} = entrance face {z_face_in:.3f}), patch S = {S} um, w0 = {w0} um")
        print(f"  monitors: exit z = {z_face_out:+.4f}, entry z = {z_face_in:+.4f}, {nfreqs} freqs, "
              f"d(nu) = {dnu/1e9:.2f} GHz <= 1/runtime = {1/runtime_ps/1e9:.2f} GHz -> window {1/dnu*1e12:.2f} ps")
        print(f"  grid: dl_min (rods) = {dl:.4f} um, actual dz inside slab min/max = "
              f"{dz_slab.min():.4f}/{dz_slab.max():.4f} um, "
              f"dx = {np.diff(sim.grid.boundaries.x).min():.4f} um, "
              f"rod diameter / dx = {2*float(np.atleast_1d(rods['minor_radius'])[0])/np.diff(sim.grid.boundaries.x).min():.1f}, "
              f"(lambda_min/n_rod)/dx = {(lambdas[1]/n_rod)/np.diff(sim.grid.boundaries.x).min():.1f}")
        print(f"  num_cells = {sim.num_cells:.3e}, num_time_steps = {sim.num_time_steps}, dt = {sim.dt:.3e} s")
        sizes = sim.monitors_data_size
        print("  monitor data size (GB): " + ", ".join(f"{k}: {v/1e9:.3f}" for k, v in sizes.items())
              + f"  -> total {sum(sizes.values())/1e9:.3f} GB per task")
        print(f"  polarisation: E along x. Bottom (entrance) layer rods along {axname[bot_or]} -> "
              f"E {'parallel' if bot_or == 0 else 'perpendicular'} to first rods; "
              f"top (exit) layer rods along {axname[top_or]} -> E {'parallel' if top_or == 0 else 'perpendicular'} to last rods")
    return sim, sim0, info


## Sanity plots (geometry from the tables)

Before anything else, build one defect structure and look at it: `sim.plot_eps(z=z_layer)` for one x-layer and one
y-layer (defect centres from the table overlaid), `sim.plot_eps(x=0)` for the stacking (alternating orientation, `d/2`
shift every second layer of the same orientation), and — **only as a check, never as input** — the same slices from the
voxelised `epsilon` stored in the file.


In [ ]:
structure_files = sorted(p for p in folder_path.rglob("*_tables.h5"))
print("\n".join(str(p) for p in structure_files))



## Main loop — build every structure, estimate (or submit)

Bookkeeping mirrors the reference and the transmission notebook: the task ids are written to
`{tidy3d root}/data/{project_name}/n_{n_rod:.2f}/{sim_name}.txt` as two lines (`id0` of the empty-box reference, then
`id` of the structure), which is the format `AM.loadFromFile(..., get_ref=True)` expects. If the `.txt` already exists
the structure is skipped ("Exist!"). `sim_name = f"{stem}_lambda_{lambdas[1]}_{lambdas[0]}"`.

With `run = False` (default) each simulation is uploaded once under a throw-away name only to obtain `web.estimate_cost`,
then deleted; nothing is started. **Running costs money — flip `run = True` deliberately.**


In [ ]:
built = {}
id0=""
for file in structure_files:
    filename = file.name
    print(f"\n=== Processing {filename}")
    sim, sim0, info = build_simulation(file,verbose=False)
    n_rod = info["n_rod"]
    sim_name = rf"{Path(filename).stem}"
    built[sim_name] = (sim, sim0, info)

    if run:
        folder_desc = rf"{Path.cwd().parents[2]}/data/{project_name}/n_{n_rod:.2f}"
        os.makedirs(folder_desc, exist_ok=True)
        txt = os.path.join(folder_desc, sim_name + ".txt")
        if os.path.exists(txt):
            print(txt); print("Exist!")
            continue
        # else:
        #      print(txt," RUN!")
        #      continue
        if ref:
            # empty-box reference, once per structure (same grid as sim)
            id0 = web.upload(sim0, folder_name=project_name, task_name=sim_name + "_0", verbose=True)
            web.start(task_id=id0)
            web.monitor(id0)
            # structure
        id = web.upload(sim, folder_name=project_name, task_name=sim_name, verbose=True)
        with open(txt, "w") as fh:
            fh.write(id0 + "\n" + id)
        web.start(task_id=id)
        web.monitor(id)
    else:
        # throw-away upload to get the cost estimate, then delete (nothing is started)
        id_test = web.upload(sim, folder_name=project_name, task_name="test_" + sim_name, verbose=False)
        try:
            cost = web.estimate_cost(id_test, verbose=False)
            print(f"  cost estimate: {cost:.3f} credits")
        finally:
            web.delete(id_test)
        print(f"  num_cells = {sim.num_cells:.3e}, num_time_steps = {sim.num_time_steps}, cylinders = {info['n_cyl']}, "
              f"dl(rods) = {info['dl']:.4f} um, monitor storage = {sum(sim.monitors_data_size.values())/1e9:.3f} GB")
        fig, axs = plt.subplots(1, 3, figsize=(24, 7), tight_layout=True)
        sim.plot(y=0, ax=axs[0]); axs[0].set_title(f"{Path(filename).stem[:40]}...  y=0")
        sim.plot(z=float(info['box'][2]/2), ax=axs[1]); axs[1].set_title("z = +t_slab_z/2 (exit monitor)")
        z_layer = float(np.max(np.atleast_1d(info["rods"]["z"])))
        sim.plot_eps(z=z_layer, freq=f0, ax=axs[2]); axs[2].set_title(f"plot_eps z = {z_layer:.3f} (top layer)")
        axs[2].set_xlim(-info['box'][0]/2, info['box'][0]/2); axs[2].set_ylim(-info['box'][1]/2, info['box'][1]/2)
        plt.show()
